# Notebook 3 — Model Architecture

This notebook defines the **shared U-Net** used in all three experiments.
The model is identical across Gaussian / Uniform / Laplace — only the noise sampled
during training changes.

Architecture: lightweight U-Net with:
- Sinusoidal time embeddings
- Residual blocks with GroupNorm
- Skip connections
- ~1.4M parameters (fast to train on MNIST)

In [1]:
!pip install -q torch torchvision matplotlib numpy

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
import os

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/diffusion_noise_project'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda


## 1. Sinusoidal Time Embedding

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    """
    Encodes the scalar timestep t into a fixed-dimensional vector.
    Uses the same sinusoidal encoding as the original Transformer / DDPM paper.
    """
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.proj = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.SiLU(),
            nn.Linear(dim * 4, dim * 4),
        )

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=t.device).float() / (half - 1)
        )
        args = t[:, None].float() * freqs[None]
        embedding = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        return self.proj(embedding)  # (B, dim*4)

## 2. Residual Block

In [ ]:
class ResBlock(nn.Module):
    """
    Residual block with:
    - Two 3×3 convolutions
    - GroupNorm normalisation
    - SiLU activation
    - Time embedding injection (as additive bias after first norm)
    - Skip connection (1×1 conv if in_ch != out_ch)
    """
    def __init__(self, in_ch, out_ch, time_emb_dim, num_groups=8):
        super().__init__()
        self.norm1 = nn.GroupNorm(min(num_groups, in_ch), in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(min(num_groups, out_ch), out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.act   = nn.SiLU()

        # Time embedding projection
        self.time_proj = nn.Linear(time_emb_dim, out_ch)

        # Skip connection
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.act(self.norm1(x))
        h = self.conv1(h)

        # Inject time embedding
        h = h + self.time_proj(self.act(t_emb))[:, :, None, None]

        h = self.act(self.norm2(h))
        h = self.conv2(h)
        return h + self.skip(x)

## 3. U-Net

In [5]:
class UNet(nn.Module):
    """
    Lightweight U-Net for MNIST noise prediction.

    Input:  (B, C, H, W) noisy image + scalar timestep t
    Output: (B, C, H, W) predicted noise eps_theta(x_t, t)

    Architecture:
        Encoder: 3 resolution levels with downsampling
        Bottleneck: 2 residual blocks
        Decoder: 3 resolution levels with upsampling + skip connections
    """
    def __init__(self, in_channels=1, model_channels=64, channel_mults=(1, 2, 4)):
        super().__init__()
        self.in_channels = in_channels
        time_emb_dim = model_channels * 4

        # Time embedding
        self.time_emb = SinusoidalTimeEmbedding(model_channels)

        # Channel widths at each level
        chs = [model_channels * m for m in channel_mults]

        # Input projection
        self.input_conv = nn.Conv2d(in_channels, chs[0], 3, padding=1)

        # Encoder
        self.enc1 = ResBlock(chs[0], chs[0], time_emb_dim)
        self.down1 = nn.Conv2d(chs[0], chs[0], 4, stride=2, padding=1)

        self.enc2 = ResBlock(chs[0], chs[1], time_emb_dim)
        self.down2 = nn.Conv2d(chs[1], chs[1], 4, stride=2, padding=1)

        self.enc3 = ResBlock(chs[1], chs[2], time_emb_dim)

        # Bottleneck
        self.mid1 = ResBlock(chs[2], chs[2], time_emb_dim)
        self.mid2 = ResBlock(chs[2], chs[2], time_emb_dim)

        # Decoder (with skip connections — note doubled input channels)
        self.up3   = nn.ConvTranspose2d(chs[2], chs[2], 4, stride=2, padding=1)
        self.dec3  = ResBlock(chs[2] + chs[1], chs[1], time_emb_dim)

        self.up2   = nn.ConvTranspose2d(chs[1], chs[1], 4, stride=2, padding=1)
        self.dec2  = ResBlock(chs[1] + chs[0], chs[0], time_emb_dim)

        self.dec1  = ResBlock(chs[0] + chs[0], chs[0], time_emb_dim)

        # Output projection
        self.out_norm = nn.GroupNorm(8, chs[0])
        self.out_act  = nn.SiLU()
        self.out_conv = nn.Conv2d(chs[0], in_channels, 1)

    def forward(self, x, t):
        """
        Args:
            x: (B, C, H, W) noisy image at timestep t
            t: (B,) integer timestep
        Returns:
            (B, C, H, W) predicted noise
        """
        # Time embedding
        t_emb = self.time_emb(t)  # (B, model_channels*4)

        # Encoder
        h0 = self.input_conv(x)
        h1 = self.enc1(h0, t_emb)
        h1d = self.down1(h1)

        h2 = self.enc2(h1d, t_emb)
        h2d = self.down2(h2)

        h3 = self.enc3(h2d, t_emb)

        # Bottleneck
        h = self.mid1(h3, t_emb)
        h = self.mid2(h, t_emb)

        # Decoder with skip connections
        h = self.up3(h)
        h = F.interpolate(h, size=h2.shape[2:], mode='nearest')  # handle odd sizes
        h = self.dec3(torch.cat([h, h2], dim=1), t_emb)

        h = self.up2(h)
        h = F.interpolate(h, size=h1.shape[2:], mode='nearest')
        h = self.dec2(torch.cat([h, h1], dim=1), t_emb)

        h = self.dec1(torch.cat([h, h0], dim=1), t_emb)

        # Output
        h = self.out_conv(self.out_act(self.out_norm(h)))
        return h

## 4. Model Inspection

In [6]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Total parameters:     {total:>10,}')
    print(f'Trainable parameters: {trainable:>10,}')
    return trainable

model = UNet(
    in_channels=1,
    model_channels=64,
    channel_mults=(1, 2, 4)
).to(DEVICE)

count_parameters(model)

Total parameters:      6,540,609
Trainable parameters:  6,540,609


6540609

In [7]:
# Forward pass sanity check
batch_size = 4
x_dummy = torch.randn(batch_size, 1, 28, 28).to(DEVICE)
t_dummy = torch.randint(0, 1000, (batch_size,)).to(DEVICE)

with torch.no_grad():
    out = model(x_dummy, t_dummy)

print(f'Input shape:  {x_dummy.shape}')
print(f'Output shape: {out.shape}')
assert out.shape == x_dummy.shape, 'Output shape mismatch!'
print('Forward pass OK.')

Input shape:  torch.Size([4, 1, 28, 28])
Output shape: torch.Size([4, 1, 28, 28])
Forward pass OK.


## 5. Training Objective
The model is trained to predict the noise eps that was added at step t.
Loss = MSE(eps_predicted, eps_true).

In [ ]:
def diffusion_loss(model, diffusion, x_0):
    """
    Compute the simplified DDPM training objective:
        L = E_{t, eps} [ || eps - eps_theta(x_t, t) ||^2 ]

    Args:
        model:     UNet
        diffusion: ForwardDiffusion (any subclass)
        x_0:       (B, C, H, W) clean images
    Returns:
        scalar loss
    """
    batch_size = x_0.shape[0]
    t = diffusion.sample_timestep(batch_size)
    x_t, eps = diffusion.q_sample(x_0, t)
    eps_pred = model(x_t, t)
    return F.mse_loss(eps_pred, eps)


print('Training objective defined: MSE(eps_predicted, eps_true)')

Training objective defined: MSE(eps_predicted, eps_true)


## 6. Reverse Process (Sampling)
DDPM ancestral sampling — used in Notebook 4 and Notebook 5 to generate images.

In [ ]:
@torch.no_grad()
def ddpm_sample(model, diffusion, n_samples=16, img_size=28, channels=1, device=DEVICE):
    """
    Generate samples using DDPM reverse process.
    Starts from pure noise (from the *training* distribution) and iteratively denoises.

    Note: during sampling we use Gaussian noise for the stochastic step
    regardless of training distribution — this is the standard DDPM sampler.
    An ablation with matched sampling noise is a good extension.
    """
    model.eval()
    shape = (n_samples, channels, img_size, img_size)

    # Start from noise
    x = torch.randn(shape, device=device)  # standard Gaussian start

    for t_val in reversed(range(diffusion.T)):
        t_batch = torch.full((n_samples,), t_val, device=device, dtype=torch.long)

        # Predict noise
        eps_pred = model(x, t_batch)

        # DDPM reverse step
        alpha_t     = diffusion.alphas[t_val]
        alpha_bar_t = diffusion.alphas_cumprod[t_val]
        beta_t      = diffusion.betas[t_val]

        # Mean of reverse posterior
        coeff = (1 - alpha_t) / (1 - alpha_bar_t).sqrt()
        mean  = (1 / alpha_t.sqrt()) * (x - coeff * eps_pred)

        if t_val > 0:
            noise = torch.randn_like(x)
            x = mean + beta_t.sqrt() * noise
        else:
            x = mean

    return x.clamp(-1, 1)


print('DDPM sampler defined.')

DDPM sampler defined.


## 7. Save Model Definition to Drive
This cell writes the model code to a `.py` file on Drive so training notebooks can import it.

In [ ]:
model_code = '''
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.proj = nn.Sequential(nn.Linear(dim, dim*4), nn.SiLU(), nn.Linear(dim*4, dim*4))
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / (half-1))
        args = t[:,None].float() * freqs[None]
        return self.proj(torch.cat([torch.sin(args), torch.cos(args)], dim=-1))

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, num_groups=8):
        super().__init__()
        self.norm1 = nn.GroupNorm(min(num_groups, in_ch), in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(min(num_groups, out_ch), out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.act = nn.SiLU()
        self.time_proj = nn.Linear(time_emb_dim, out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, t_emb):
        h = self.act(self.norm1(x))
        h = self.conv1(h)
        h = h + self.time_proj(self.act(t_emb))[:,:,None,None]
        h = self.act(self.norm2(h))
        h = self.conv2(h)
        return h + self.skip(x)

class UNet(nn.Module):
    def __init__(self, in_channels=1, model_channels=64, channel_mults=(1,2,4)):
        super().__init__()
        time_emb_dim = model_channels * 4
        self.time_emb = SinusoidalTimeEmbedding(model_channels)
        chs = [model_channels * m for m in channel_mults]
        self.input_conv = nn.Conv2d(in_channels, chs[0], 3, padding=1)
        self.enc1 = ResBlock(chs[0], chs[0], time_emb_dim)
        self.down1 = nn.Conv2d(chs[0], chs[0], 4, stride=2, padding=1)
        self.enc2 = ResBlock(chs[0], chs[1], time_emb_dim)
        self.down2 = nn.Conv2d(chs[1], chs[1], 4, stride=2, padding=1)
        self.enc3 = ResBlock(chs[1], chs[2], time_emb_dim)
        self.mid1 = ResBlock(chs[2], chs[2], time_emb_dim)
        self.mid2 = ResBlock(chs[2], chs[2], time_emb_dim)
        self.up3 = nn.ConvTranspose2d(chs[2], chs[2], 4, stride=2, padding=1)
        self.dec3 = ResBlock(chs[2]+chs[1], chs[1], time_emb_dim)
        self.up2 = nn.ConvTranspose2d(chs[1], chs[1], 4, stride=2, padding=1)
        self.dec2 = ResBlock(chs[1]+chs[0], chs[0], time_emb_dim)
        self.dec1 = ResBlock(chs[0]+chs[0], chs[0], time_emb_dim)
        self.out_norm = nn.GroupNorm(8, chs[0])
        self.out_act = nn.SiLU()
        self.out_conv = nn.Conv2d(chs[0], in_channels, 1)
    def forward(self, x, t):
        t_emb = self.time_emb(t)
        h0 = self.input_conv(x)
        h1 = self.enc1(h0, t_emb); h1d = self.down1(h1)
        h2 = self.enc2(h1d, t_emb); h2d = self.down2(h2)
        h3 = self.enc3(h2d, t_emb)
        h = self.mid2(self.mid1(h3, t_emb), t_emb)
        h = F.interpolate(self.up3(h), size=h2.shape[2:], mode="nearest")
        h = self.dec3(torch.cat([h, h2], dim=1), t_emb)
        h = F.interpolate(self.up2(h), size=h1.shape[2:], mode="nearest")
        h = self.dec2(torch.cat([h, h1], dim=1), t_emb)
        h = self.dec1(torch.cat([h, h0], dim=1), t_emb)
        return self.out_conv(self.out_act(self.out_norm(h)))
'''

model_path = os.path.join(PROJECT_DIR, 'unet.py')
with open(model_path, 'w') as f:
    f.write(model_code)
print(f'Model saved to {model_path}')
print('Notebook 3 complete. Proceed to Notebooks 4a/b/c for training.')

Model saved to /content/drive/MyDrive/diffusion_noise_project/unet.py
Notebook 3 complete. Proceed to Notebooks 4a/b/c for training.
